# Chương 3: FKG-UM / FKG-MM trên BRSET 16.266 ảnh, FKG-UM (Bảng) dùng 9 thuộc tính bảng của FKG-MM

Bộ mã `FKGMM_BRSET_Chuong3` (bản `FKGUM9`), **không cần GPU** (Runtime CPU là đủ).

**Điểm mới:** FKG-UM (Bảng) dùng nhóm dữ liệu `table_sel`, chọn thuộc tính bảng **đúng như nhánh bảng của FKG-MM (Φsel)**
(chuẩn hoá z, ANOVA, chỉ fit trên fold huấn luyện). Mặc định `K_TABLE_UM = 9`, tức trùng đúng 9 thuộc tính bảng của FKG-MM trong từng fold:

| `K_TABLE_UM` | Thuộc tính FKG-UM (Bảng) dùng |
|---|---|
| 7 | 7 thuộc tính xếp hạng ANOVA cao nhất trong 9 thuộc tính bảng FKG-MM chọn |
| **9 (mặc định)** | Trùng đúng 9 thuộc tính bảng của FKG-MM trong từng fold (so sánh công bằng nhất) |

LR / RF / XGBoost / MLP trên dữ liệu bảng và Late Fusion **vẫn dùng 13 cột** như cũ, nên các dòng đối chứng của
Bảng 3.1 không đổi; chỉ dòng FKG-UM (Bảng) (và Tabular-only ở bảng cắt bỏ) thay đổi.

**Chuẩn bị:** tải `FKGMM_BRSET_Chuong3_FKGUM9.zip` lên `MyDrive/`, chạy lần lượt các ô. Kết quả và log nằm trên Drive;
phiên bị ngắt thì chạy lại từ đầu, phần đã xong được bỏ qua (`--resume`). Thời gian tham khảo: dữ liệu khoảng 5 phút,
mỗi lượt kịch bản khoảng 35 phút trên máy 8 nhân (Colab 2 nhân có thể lâu hơn).

In [ ]:
from pathlib import Path
MODE = "full"                      # "full" | "smoke" (dữ liệu tổng hợp, vài phút, KHÔNG phải kết quả BRSET)
ZIP_PATH = Path("/content/drive/MyDrive/FKGMM_BRSET_Chuong3_FKGUM9.zip")
OUTPUT_ROOT = Path("/content/drive/MyDrive/FKGMM_BRSET_chuong3_FKGUM")
K_TABLE_UM_LIST = [9]              # 9 = đúng 9 thuộc tính bảng của FKG-MM; có thể thêm giá trị khác, mỗi giá trị một lượt
SCENARIOS = ["M1", "M2", "M3", "M4", "M5", "M6", "EPS"]   # có thể rút gọn, ví dụ ["M1", "M4"] để chạy nhanh
RUN_DEDUP = False                  # biến thể lọc luật trùng/mâu thuẫn (không cần cho bảng 3.1)
WORKDIR = Path("/content")

## 1. Gắn Drive, giải nén, cài thư viện, chạy test

In [ ]:
import hashlib, json, subprocess, sys, time, zipfile
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Không chạy trên Colab: bỏ qua bước gắn Drive")
SMOKE = MODE == "smoke"
if SMOKE:
    OUTPUT_ROOT = OUTPUT_ROOT.with_name(OUTPUT_ROOT.name + "_SMOKE")
assert ZIP_PATH.exists(), f"Chưa thấy {ZIP_PATH}: tải FKGMM_BRSET_Chuong3_FKGUM9.zip lên Drive"
with zipfile.ZipFile(ZIP_PATH) as z:
    assert all((WORKDIR / n).resolve().is_relative_to(WORKDIR.resolve()) for n in z.namelist()), "Đường dẫn lạ trong zip"
    z.extractall(WORKDIR)
bundle = WORKDIR / "FKGMM_BRSET_Chuong3"
assert (bundle / "chuong3.py").read_text(encoding="utf-8").count("table_sel") > 0, "Zip không phải bản FKGUM9"
print("ZIP sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
LOGS = OUTPUT_ROOT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run_logged(args, log_name):
    # Chạy lệnh trong gói, in log ra notebook và ghi đầy đủ lên Drive; lỗi thì dừng ô.
    command = [sys.executable, "-B", "-X", "utf8", "-u", *map(str, args)]
    print("Lệnh:", " ".join(command), flush=True)
    with open(LOGS / log_name, "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(command)}\n")
        process = subprocess.Popen(command, cwd=bundle, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in process.stdout:
            log.write(line)
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Lệnh lỗi (mã {code}); xem {LOGS / log_name}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(bundle / "requirements.txt")], check=True)
# chạy trực tiếp từ thư mục tests/ (tránh trùng tên với gói `tests` có sẵn trong site-packages của Colab)
run_logged(["-m", "unittest", "discover", "-s", "tests", "-p", "test_ch3.py", "-v"], "tests.log")

## 2. Tạo cấu hình cho từng giá trị `K_TABLE_UM`

Mỗi giá trị có một file cấu hình và một thư mục kết quả riêng (`k7/`, `k9/`), vì mã từ chối ghi đè kết quả của cấu hình khác.

In [ ]:
base_cfg = json.loads((bundle / "config_ch3.json").read_text(encoding="utf-8"))
RUNS = {}
for k in K_TABLE_UM_LIST:
    assert 1 <= k <= base_cfg["k_tab"], f"K_TABLE_UM = {k} phải nằm trong 1..{base_cfg['k_tab']} (số thuộc tính bảng của FKG-MM)"
    cfg = {**base_cfg, "k_table_um": k, "fkg_um_table": "table_sel"}
    cfg_path = bundle / f"config_ch3_k{k}.json"
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    RUNS[k] = {"config": cfg_path, "out": OUTPUT_ROOT / f"k{k}" / "chuong3", "out_dedup": OUTPUT_ROOT / f"k{k}" / "chuong3_dedup"}
    print(f"K_TABLE_UM = {k}: cấu hình {cfg_path.name}, kết quả {RUNS[k]['out']}")

## 3. Dựng dữ liệu và chạy các kịch bản

Dựng 7 nhóm dữ liệu gốc và nhóm `table_sel` cho 5 fold; sau đó chạy các kịch bản trong `SCENARIOS` với FISA-GM-w.

In [ ]:
for k, r in RUNS.items():
    print(f"\n==================== K_TABLE_UM = {k} ====================", flush=True)
    run_logged(["chuong3.py", "all", "--config", r["config"], "--out", r["out"], "--resume",
                "--scenarios", *SCENARIOS, *(["--smoke"] if SMOKE else [])], f"chuong3_k{k}.log")
    if RUN_DEDUP:
        run_logged(["chuong3.py", "run", "--config", r["config"], "--out", r["out_dedup"], "--data", r["out"] / "data",
                    "--rule-base", "dedup_conf", "--resume", "--scenarios", *SCENARIOS], f"chuong3_dedup_k{k}.log")

## 4. Kiểm tra thuộc tính FKG-UM (Bảng) đã dùng

Đối chiếu từng fold: thuộc tính của `table_sel` phải nằm trong tập thuộc tính bảng FKG-MM chọn (và trùng hẳn khi K = 9).

In [ ]:
import pandas as pd
def selected(out, group, fold):
    s = pd.read_csv(out / "data" / group / f"fold_{fold:02d}" / "selected_features.csv")
    s = s[s["selected"].astype(str).str.lower() == "true"]
    if "branch" in s:
        s = s[s["branch"] == "table"]
    return list(s["source_column"])
for k, r in RUNS.items():
    rows = []
    for fold in range(1, 6):
        um, mm = selected(r["out"], "table_sel", fold), selected(r["out"], "fusion", fold)
        rows.append({"fold": fold, "FKG-UM (Bảng)": ", ".join(um), "số TT": len(um),
                     "⊆ thuộc tính bảng FKG-MM": set(um) <= set(mm), "trùng hẳn": set(um) == set(mm)})
    print(f"\nK_TABLE_UM = {k}")
    display(pd.DataFrame(rows))

## 5. So sánh FKG-UM (Bảng) với FKG-MM và bản cũ 13 cột

In [ ]:
from IPython.display import Image, Markdown, display
metrics_cols = {"acc": "Acc", "f1": "F1", "auc_roc": "AUC", "pr_auc_trap": "PR-AUC", "sens": "Độ nhạy", "spec": "Độ đặc hiệu", "balacc": "BalAcc"}
def summary(df, model):
    d = df[(df.scenario == "M1") & (df.model == model) & (df.data == "FKG")]
    cols = [c for c in metrics_cols if c in d]
    return {metrics_cols[c]: f"{100 * d[c].mean():.1f} ± {100 * d[c].std(ddof=1):.1f}" for c in cols} | {"số fold": len(d)}
table = []
for k, r in RUNS.items():
    pf = r["out"] / "ket_qua" / "per_fold.csv"
    if pf.exists():
        df = pd.read_csv(pf)
        table.append({"Mô hình": f"FKG-UM (Bảng), {k} thuộc tính", **summary(df, "FKG-UM (Bảng)")})
        table.append({"Mô hình": f"FKG-MM (đề xuất) [lượt k = {k}]", **summary(df, "FKG-MM (đề xuất)")})
display(pd.DataFrame(table))
print("Bản cũ trong luận án (13 cột): FKG-UM (Bảng) AUC 92,7 ± 1,4; BalAcc 83,7 ± 0,8; FKG-MM AUC 92,5 ± 1,3; BalAcc 85,3 ± 1,1")
for k, r in RUNS.items():
    md_path = r["out"] / "ket_qua" / "ket_qua.md"
    if md_path.exists():
        text = md_path.read_text(encoding="utf-8")
        start = text.find("### KB-M1")
        display(Markdown(f"---\n### K_TABLE_UM = {k}: kiểm định FKG-MM − FKG-UM (Bảng)\n\n" + text[start:text.find("### KB-M2")]))

## 6. Xem toàn bộ kết quả và hình

In [ ]:
for k, r in RUNS.items():
    for title, path in ((f"K = {k}, cơ sở luật đầy đủ", r["out"] / "ket_qua/ket_qua.md"),
                        (f"K = {k}, lọc luật", r["out_dedup"] / "ket_qua/ket_qua.md")):
        if path.exists():
            display(Markdown(f"---\n## {title}\n\n" + path.read_text(encoding="utf-8")))
    for name in ("roc_pr_kb_m1_m2.png", "roc_pr_kb_m5.png", "ap_theo_fold.png"):
        path = r["out"] / "ket_qua/hinh" / name
        if path.exists():
            display(Markdown(f"**K = {k}: {name}**"))
            display(Image(filename=str(path)))

## 7. Đóng gói để gửi lại

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M")
package = OUTPUT_ROOT.parent / f"{OUTPUT_ROOT.name}_bao_cao_{stamp}.zip"
paths = [*LOGS.rglob("*")]
for r in RUNS.values():
    paths += [r["config"], r["out"] / "data/prepare_manifest.json"]
    for base in (r["out"], r["out_dedup"]):
        paths += [*(base / "ket_qua").rglob("*")]                  # gồm hinh/ và predictions/
    paths += [*(r["out"] / "data").glob("*/fold_*/metadata.json"), *(r["out"] / "data").glob("*/fold_*/selected_features.csv")]
with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
    for path in paths:
        if path.is_file():
            arc = path.relative_to(OUTPUT_ROOT) if path.is_relative_to(OUTPUT_ROOT) else Path("config") / path.name
            z.write(path, arc)
print("Gửi file:", package, f"({package.stat().st_size / 2**20:.1f} MiB)")